In [1]:
"""Load the local chunker and discover the sample directory."""

import logging
from html import escape
from pathlib import Path

import ipywidgets as widgets
import tiktoken
from IPython.display import HTML, Markdown, display

from ingestra.chunker import Chunk, MarkdownTokenChunker

logger = logging.getLogger(__name__)


def find_repository_root(start: Path) -> Path:
    """Find the repository above the kernel's working directory.

    Args:
        start: Directory from which to search upward.

    Returns:
        Repository directory containing the package and project manifest.

    Raises:
        FileNotFoundError: The kernel was started outside the repository.
    """
    directory = start.resolve()
    for candidate in (directory, *directory.parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "src" / "ingestra").is_dir():
            return candidate
    raise FileNotFoundError("Start the notebook kernel from the ingestra repository or one of its subdirectories")


ROOT: Path = find_repository_root(Path.cwd())
SAMPLES_DIRECTORY: Path = ROOT / "exeperiment" / "chunkers" / "samples"
SAMPLE_PATHS: list[Path] = sorted(SAMPLES_DIRECTORY.glob("*.md"))
if not SAMPLE_PATHS:
    raise FileNotFoundError(f"No Markdown samples found in {SAMPLES_DIRECTORY}")
ENCODING: tiktoken.Encoding = tiktoken.get_encoding("cl100k_base")

display(Markdown(f"Found **{len(SAMPLE_PATHS)} samples** in `{SAMPLES_DIRECTORY}`."))


Found **12 samples** in `C:\Work\personal-workspace\Personal-Projects\ingestra\exeperiment\chunkers\samples`.

In [2]:
"""Create verified chunks and reusable rendered or raw text panels."""


def chunk_sample(sample_path: Path, *, chunk_size: int = 2048) -> tuple[str, list[Chunk]]:
    """Read a UTF-8 sample and verify its source and embedding-token contracts.

    Args:
        sample_path: Local Markdown file to read without normalizing newlines.
        chunk_size: Soft token-size goal; natural boundaries and structural context can exceed it.

    Returns:
        Original document and ordered chunks with independently checked token counts.

    Raises:
        OSError: The sample cannot be read.
        UnicodeDecodeError: The sample is not UTF-8.
        TypeError: The token size has an unsupported type.
        ValueError: The size is invalid or cannot fit one source character.
        RuntimeError: A chunk violates the source or embedding-token contract.
    """
    chunker = MarkdownTokenChunker(chunk_size=chunk_size)
    logger.info("Chunking sample '%s' with chunk_size=%d tokens", sample_path.name, chunk_size)
    markdown = sample_path.read_bytes().decode("utf-8")
    chunks = chunker.chunk(markdown)
    if markdown.strip() and "".join(chunk.original_text for chunk in chunks) != markdown:
        raise RuntimeError(f"Source reconstruction failed for {sample_path.name}")
    for number, chunk in enumerate(chunks, start=1):
        if not chunk.original_text or not chunk.embedding_text or chunk.token_count is None:
            raise RuntimeError(f"Chunk {number} has missing source, embedding text, or token count")
        measured_tokens = len(ENCODING.encode_ordinary(chunk.embedding_text))
        if chunk.token_count != measured_tokens:
            raise RuntimeError(f"Chunk {number} reports an incorrect embedding token count")
    logger.info("Chunked sample '%s' into %d verified chunks", sample_path.name, len(chunks))
    return markdown, chunks


def text_panel(title: str, text: str, *, raw: bool) -> widgets.VBox:
    """Build one comparison column without changing the supplied text.

    Args:
        title: Label above the text.
        text: Original source or complete embedding input.
        raw: Whether to show escaped source instead of rendered Markdown.

    Returns:
        A responsive panel containing the label and displayed text.
    """
    output = widgets.Output(layout=widgets.Layout(width="100%"))
    with output:
        if raw:
            display(HTML(f'<pre style="white-space: pre-wrap; overflow-wrap: anywhere;">{escape(text)}</pre>'))
        else:
            display(Markdown(text))
    label = widgets.HTML(value=f"<strong>{escape(title)}</strong>")
    layout = widgets.Layout(width="49%", min_width="280px", border="1px solid #888", padding="10px")
    return widgets.VBox([label, output], layout=layout)


In [ ]:
"""Select a sample and render its source and embedding inputs side by side."""

sample_selector = widgets.Dropdown(options=[(path.stem.replace("_", " "), path.name) for path in SAMPLE_PATHS], description="Sample:", layout=widgets.Layout(width="100%"))
chunk_size_input = widgets.IntText(value=2048, description="Chunk size (tokens):", style={"description_width": "initial"})
size_hint = widgets.HTML(value="<p>One soft token goal. Natural boundaries and structural context can exceed it; prose overlap is not added.</p>")
raw_view = widgets.Checkbox(value=False, description="Raw Markdown")
show_document = widgets.Checkbox(value=True, description="Show full sample")
create_button = widgets.Button(description="Create chunks", button_style="primary", icon="scissors")
results = widgets.Output()


@results.capture(clear_output=True, wait=True)
def render_selected_sample(_: widgets.Button | None = None) -> None:
    """Render the selected sample; show callback errors in the results area.

    Args:
        _: Optional button supplied by the widget callback.
    """
    selected = sample_selector.value
    if not isinstance(selected, str):
        raise TypeError("Select a Markdown sample before creating chunks")
    sample_path = SAMPLES_DIRECTORY / selected
    markdown, chunks = chunk_sample(sample_path, chunk_size=chunk_size_input.value)
    source_tokens = len(ENCODING.encode_ordinary(markdown))
    largest_chunk = max((chunk.token_count for chunk in chunks if chunk.token_count is not None), default=0)
    source_status = "verified" if markdown.strip() else "not applicable to blank input"
    display(Markdown(f"## {sample_path.name}\n\n**{len(chunks)} chunks** | Source: **{source_tokens} tokens** | Token goal: **{chunk_size_input.value}** | Largest embedding: **{largest_chunk} tokens**\n\nSource reconstruction: **{source_status}**. All embedding token counts were independently checked; the goal is not a hard ceiling."))

    for number, chunk in enumerate(chunks, start=1):
        if chunk.embedding_text is None:
            raise RuntimeError(f"Chunk {number} has no embedding text to display")
        pages = ", ".join(str(page) for page in chunk.page_numbers) or "none"
        display(Markdown(f"### Chunk {number}\n\nEmbedding: **{chunk.token_count} tokens** | Pages: **{pages}** | Source: **{len(chunk.original_text)} characters** | Goal: **{chunk_size_input.value} tokens**"))
        original_panel = text_panel("Original text", chunk.original_text, raw=raw_view.value)
        embedding_panel = text_panel("Embedding-ready text", chunk.embedding_text, raw=raw_view.value)
        row_layout = widgets.Layout(flex_flow="row wrap", justify_content="space-between", align_items="stretch", width="100%")
        display(widgets.HBox([original_panel, embedding_panel], layout=row_layout))


create_button.on_click(render_selected_sample)
view_row = widgets.HBox([raw_view, show_document], layout=widgets.Layout(flex_flow="row wrap"))
display(widgets.VBox([sample_selector, chunk_size_input, size_hint, view_row, create_button, results]))
render_selected_sample()
